# FRAMEWORK

In this file a framework for feature engineering is developed.

Techniques implemented:
- Time Series Irregularity
  - Re-sampling                                             [x]
  - Interpolation (shouldn't this be missing values??)
- Missing values
  - Forward Propagation                                     [x]
  - Backward Propagation                                    [x]
  - Statistical Inferences
- Feature selection
  - Filter methods
    - PCA                                                   [x]
    - Correlation matrix [x]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

class TimeSeriesProcessor:
    def __init__(self, df, time_col='DateTime', id_col='ProcessId', pred_col='event'):
        self.df = df.copy()
        self.time_col = time_col
        self.id_col = id_col
        self.pred_col = pred_col

    def regularity_resample(self, freq='2H'):
        # Ensure the index columns are unique
        self.df = self.df.drop_duplicates(subset=[self.id_col, self.time_col])

        resampled_dfs = []
        for process_id, group in self.df.groupby(self.id_col):
            # Create a full index with the desired frequency for each process
            full_index = pd.MultiIndex.from_product(
                [[process_id], 
                 pd.date_range(start=group[self.time_col].min(), 
                               end=group[self.time_col].max(), 
                               freq=freq)],
                names=[self.id_col, self.time_col]
            )

            # Set the index and reindex to get a regular time series for each ProcessId
            group.set_index([self.id_col, self.time_col], inplace=True)
            group = group.reindex(full_index)

            # Interpolate missing values for numeric columns
            numeric_cols = group.select_dtypes(include=[np.number]).columns
            group[numeric_cols] = group[numeric_cols].interpolate()

            resampled_dfs.append(group)

        # Concatenate all resampled DataFrames
        self.df = pd.concat(resampled_dfs).reset_index()

        return self.df

    def NA_interpolate(self, method='linear'):
        """
        Interpolate missing values using the specified method.
        """
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        self.df[numeric_cols] = (
            self.df.groupby(self.id_col)[numeric_cols]
            .apply(lambda g: g.interpolate(method=method))
            .reset_index(level=0, drop=True)  # Reset index to align with self.df
        )
        return self.df


    def NA_forward_fill(self):
        """
        Forward propagate missing values.
        """
        self.df = self.df.groupby(self.id_col).ffill()
        return self.df

    def NA_backward_fill(self):
        """
        Backward propagate missing values.
        """
        self.df = self.df.groupby(self.id_col).bfill()
        return self.df

    def NA_statistical_impute(self, strategy='mean'):
        """
        Impute missing values using statistical methods.
        """
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        if strategy == 'mean':
            self.df[numeric_cols] = self.df.groupby(self.id_col)[numeric_cols].transform('mean')
        elif strategy == 'median':
            self.df[numeric_cols] = self.df.groupby(self.id_col)[numeric_cols].transform('median')
        return self.df

    def plot_resampling_distance(self, before, after):
        process_ids = before[self.id_col].unique()
        
        for process_id in process_ids:
            fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

            # Plot before resampling
            group_before = before[before[self.id_col] == process_id]
            axes[0].scatter(group_before[self.time_col], [1] * len(group_before), label=f'Process {process_id}')
            axes[0].set_title(f'Before Resampling - Process {process_id}')
            axes[0].set_ylabel('Presence')
            axes[0].legend()

            # Plot after resampling
            group_after = after[after[self.id_col] == process_id]
            axes[1].scatter(group_after[self.time_col], [1] * len(group_after), label=f'Process {process_id}')
            axes[1].set_title(f'After Resampling - Process {process_id}')
            axes[1].set_ylabel('Presence')
            axes[1].legend()

            plt.xlabel('Timestamp')
            plt.show()

    def feature_selection_by_correlation(self, threshold=0.9, plot=False):
        numeric_df = self.df.select_dtypes(include=[np.number])
        corr_matrix = numeric_df.corr()

        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
        plt.title("Correlation Matrix Before Feature Selection")
        #plt.show()

        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        drop_features = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

        self.df.drop(columns=drop_features, inplace=True)

        if plot:
            new_corr_matrix = self.df.select_dtypes(include=[np.number]).corr()
            plt.figure(figsize=(10, 8))
            sns.heatmap(new_corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
            plt.title("Correlation Matrix After Feature Selection")
            plt.show()

        return self.df

    def feature_selection_by_pca(self, variance_threshold=0.95, exclude_columns=None, plot=False):
        """
        Reduces the numeric attributes using PCA while excluding specified columns.
        
        Parameters:
        variance_threshold: float, cumulative explained variance threshold.
        exclude_columns: list of columns to exclude from PCA transformation (e.g., primary keys, target variable).
        plot: boolean, whether to plot the explained variance by principal components.

        Returns:
        Updated DataFrame with principal components replacing the reduced numeric features.
        """
        if exclude_columns is None:
            exclude_columns = [self.time_col, self.id_col, self.pred_col]  # default excluded columns

        # Identify numeric columns that are candidates for PCA by excluding specified columns.
        numeric_cols = [col for col in self.df.select_dtypes(include=[np.number]).columns if col not in exclude_columns]

        # Select only the attributes to reduce.
        X = self.df[numeric_cols]
        # Apply PCA to find the explained variance.
        pca = PCA()
        X_pca = pca.fit_transform(X)
        explained_variance_ratio = np.cumsum(pca.explained_variance_ratio_)

        if plot:
            # Plot the explained variance.
            plt.figure(figsize=(10, 5))
            plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', linestyle='--')
            plt.axhline(y=variance_threshold, color='r', linestyle='-')
            plt.xlabel('Number of Principal Components')
            plt.ylabel('Cumulative Explained Variance')
            plt.title('Explained Variance by Principal Components')
            plt.show()

        num_components = np.argmax(explained_variance_ratio >= variance_threshold) + 1
        print(f"Selecting {num_components} principal components that explain {variance_threshold * 100}% of variance")

        # Perform PCA with the determined number of components.
        pca = PCA(n_components=num_components)
        X_reduced = pca.fit_transform(X)

        # Append the principal components as new columns.
        for i in range(num_components):
            self.df[f'PC{i+1}'] = X_reduced[:, i]

        # Drop the original numeric features that were reduced.
        self.df.drop(columns=numeric_cols, inplace=True)
        
        return self.df

    def feature_extraction_add_lag_features(self, features=None, n_lags=2, method='shift'):
        """
        Adds lag-based features to the dataframe for the given features.
        
        Parameters:
            features: list of features to generate lag features for.
                    If None, all numeric columns (excluding key columns) are used.
            n_lags: Number of lags to consider.
            method: 'shift' for simple shifting (one column per lag) 
                    or 'aggregate' to compute the mean of the last n lag values and create a single feature.
                    
        Returns:
            Updated dataframe with lag features (excluding first n_lags rows per processId).
        """
        df = self.df.copy()

        if features is None:
            # Select numeric columns excluding ID and time columns
            features = df.select_dtypes(include=[np.number]).columns.tolist()
            features = [col for col in features if col not in [self.id_col, self.time_col, self.pred_col]]

        # Ensure data is sorted by id and time for proper lag calculation
        df.sort_values(by=[self.id_col, self.time_col], inplace=True)

        # Store the processed groups
        processed_groups = []

        for process_id, group in df.groupby(self.id_col):
            group = group.sort_values(by=self.time_col)  # Ensure order
            
            if method == 'shift':
                for feature in features:
                    for lag in range(1, n_lags + 1):
                        group[f"{feature}_lag{lag}"] = group[feature].shift(lag)

            elif method == 'aggregate':
                for feature in features:
                    lag_mean_col = f"{feature}_lag_mean_{n_lags}"
                    group[lag_mean_col] = (
                        group[feature]
                        .shift(1)  # Shift to avoid leakage
                        .rolling(window=n_lags, min_periods=1)
                        .mean()
                    )

            else:
                raise ValueError("Method not recognized. Use 'shift' or 'aggregate'.")

            # Drop the first `n_lags` rows for this processId
            group = group.iloc[n_lags:]

            processed_groups.append(group)

        # Concatenate processed groups
        df_result = pd.concat(processed_groups).reset_index(drop=True)

        return df_result


In [ ]:
df = pd.read_pickle("Datasets/final_dataset.pkl")
processor = TimeSeriesProcessor(df)

# Resampling
#df_resampled = processor.regularity_resample(freq='5T')

# Interpolation
#df_resampled = processor.NA_interpolate(method='linear')

# Feature Extraction
#df_resampled = processor.feature_extraction_add_lag_features(n_lags=2, method='aggregate')

# Feature Selection
df_resampled = processor.feature_selection_by_pca(variance_threshold=0.95)

df_resampled.head()


In [ ]:
df_resampled.columns

In [ ]:
#########################################
#  F E A T U R E   E X T R A C T I O N  #
#########################################

# df_with_lags = processor.feature_extraction_add_lag_features(n_lags=2, method='aggregate')
# df_with_lags[['ProcessId', 'DateTime','TPressHardSample1','TPressHardSample1_lag_mean_2']].head()

In [ ]:
#########################################
#  F E A T U R E   S E L E C T I O N    #
#########################################

#df_no_corr = processor.feature_selection_by_correlation(threshold=0.9)
#df_pca = processor.feature_selection_by_pca(variance_threshold=0.99)

In [ ]:
import itertools

class TimeSeriesProcessorExhaustive:
    def __init__(self, df, time_col='DateTime', id_col='ProcessId'):
        self.df = df
        self.time_col = time_col
        self.id_col = id_col

    def apply_pipeline(self, step_funcs):
        """Apply a single combination of steps to process the DataFrame."""
        # Always work with a fresh copy of the original dataframe
        temp_df = self.df.copy()
        
        for func_name in step_funcs:
            if func_name:
                params = self.param_options.get(func_name, {})
                param_combinations = [dict(zip(params.keys(), values)) for values in itertools.product(*params.values())] or [{}]

                for param_set in param_combinations:
                    processor = TimeSeriesProcessor(self.df.copy())  # Always start with a fresh copy
                    method = getattr(processor, func_name)
                    print(f"Applying {func_name} with params: {param_set}")

                    temp_df = method(**param_set) if param_set else method()

                    # TODO keep the "best one" ???? how to work around this issue

        return temp_df

    def exhaustive_search(self):
        """Perform an exhaustive search over all function combinations."""
        step_prefixes = ['regularity', 'NA', 'feature_extraction', 'feature_selection']
        functions_by_step = {prefix: [] for prefix in step_prefixes}

        self.param_options = {
            'regularity_resample': {'freq': ['30T', '1H', '2H', '4H', '8H', '12H', '1D']},
            'NA_interpolate': {'method': ['linear', 'quadratic', 'cubic', 'nearest']},
            'NA_forward_fill': {},
            'NA_backward_fill': {},
            'NA_statistical_impute': {'strategy': ['mean', 'median']},
            'feature_extraction_add_lag_features': {'n_lags': [1, 2, 3, 5, 7], 'method': ['aggregate']},
            'feature_selection_by_correlation': {'threshold': [0.8, 0.85, 0.9, 0.95]},
            'feature_selection_by_pca': {'variance_threshold': [0.85, 0.9, 0.95, 0.99]},
        }
        
        for func_name in self.param_options.keys():
            for prefix in step_prefixes:
                if func_name.startswith(prefix):
                    functions_by_step[prefix].append(func_name)

        # Now generate the combinations using the function names from param_options
        step_combinations = list(itertools.product(*[[None] + functions_by_step[prefix] for prefix in step_prefixes]))
        

        results_dict = {}
        for step_funcs in step_combinations:
            print(f"Processing combination: {step_funcs}")
            processed_df = self.apply_pipeline(step_funcs)
            results_dict[step_funcs] = processed_df

        return results_dict


In [ ]:
import os
def merge_and_save_datasets(train_dict, test_dict, output_folder="DatasetCleaned"):
    """Merge train & test datasets based on function pipelines and save the results."""
    os.makedirs(output_folder, exist_ok=True)

    for key in train_dict.keys():
        if key in test_dict:
            df_train = train_dict[key]
            df_test = test_dict[key]

            # Ensure column consistency
            common_columns = df_train.columns.intersection(df_test.columns)
            df_train = df_train[common_columns]
            df_test = df_test[common_columns]

            # Merge datasets
            final_df = pd.concat([df_train, df_test], ignore_index=True)
            final_df.sort_values(by=["ProcessId", "DateTime"], inplace=True)
            final_df.reset_index(drop=True, inplace=True)

            # Save file
            filename = f"{output_folder}/final_dataset_{'_'.join(filter(None, key))}.csv"
            final_df.to_csv(filename, index=False)

    print(f"✅ All datasets saved in '{output_folder}'!")

In [ ]:
def train_test_split_by_time(df, time_col='DateTime', id_col='ProcessId', train_ratio=0.7):
    """
    Splits the dataset into training and testing sets based on time.
    
    Parameters:
        df (pd.DataFrame): The input dataset.
        time_col (str): The name of the datetime column.
        id_col (str): The process identifier column.
        train_ratio (float): Proportion of oldest timestamps to use for training.
    
    Returns:
        train_df (pd.DataFrame): Training set (oldest 70% of timestamps per ProcessId).
        test_df (pd.DataFrame): Testing set (newest 30% of timestamps per ProcessId).
    """
    train_list = []
    test_list = []

    for process_id, group in df.groupby(id_col):
        group = group.sort_values(by=time_col)  # Ensure chronological order
        split_idx = int(len(group) * train_ratio)  # Determine the split index
        train_list.append(group.iloc[:split_idx])  # Oldest 70% for training
        test_list.append(group.iloc[split_idx:])  # Newest 30% for testing

    train_df = pd.concat(train_list).reset_index(drop=True)
    test_df = pd.concat(test_list).reset_index(drop=True)

    return train_df, test_df

In [ ]:
df = pd.read_pickle("Datasets/final_dataset.pkl")
# df.describe()

In [ ]:
train, test = train_test_split_by_time(df)

search_train = TimeSeriesProcessorExhaustive(train)
search_test = TimeSeriesProcessorExhaustive(test)

train_results = search_train.exhaustive_search()
test_results = search_test.exhaustive_search()

merge_and_save_datasets(train_results, test_results)

## Duvidas

- pca estará bem feito? só encontra 1 principal component? meio inútil??
- No SOTA, tenho que interpolation é uma tecnica de 'irregularity' mas n seria missing values?
- O interpolation apenas se aplica a colunas numéricas, será que a framework tem de decidir outra forma de imputação para essas colunas ou assumimos que por fora é preciso transformar:
        feature
        a
        b
        a
    em 
        feature_a feature_b
           1          0
           0          1
           1          0
- combinações da pesquisa exaustiva: o pca n aceita null values, então n faz sentido ter a opção None e feature_selection_by_pca. será que se descartaria a opção None no stage NA???